# 1. ClickHouse DDL a partir de tracker_raw

Objetivo: criar apenas databases e tabelas vazias (raw, trusted, gold). Sem ingestao, sem leitura Oracle, sem regras de negocio.

Fonte de metadados: tabela `tracker_raw` (default.tracker_raw ou TRACKER_RAW_DATABASE.TRACKER_RAW_TABLE). Schema esperado: system, table_origin, column_name, column_type, ordinal_position, is_natural_key, gold_table_name (opcional).

# 2. Configuracao e Spark Session

In [233]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _root = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd()
    for p in [Path.cwd(), _root, Path.cwd().parent]:
        env_path = p / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            break
except Exception:
    pass

CH_HOST = os.getenv("CLICKHOUSE_HOST", "")
CH_PORT = int(os.getenv("CLICKHOUSE_PORT", "8443"))
CH_USER = os.getenv("CLICKHOUSE_USER", "default")
CH_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CH_DEFAULT_DB = os.getenv("CLICKHOUSE_DATABASE", "default")
TRACKER_DB = os.getenv("TRACKER_RAW_DATABASE", CH_DEFAULT_DB)
TRACKER_TABLE = os.getenv("TRACKER_RAW_TABLE", "tracker_raw")

In [234]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("clickhouse-schema-ddl")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# 3. Cliente ClickHouse para DDL

In [235]:
import clickhouse_connect

ch_client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=CH_PORT,
    username=CH_USER,
    password=CH_PASSWORD,
    secure=os.getenv("CLICKHOUSE_SECURE", "true").lower() == "true",
)
ch_client.command("SELECT 1")

1

# 4. Criar databases raw, trusted, gold

In [236]:
for db in ["raw", "trusted", "gold"]:
    ch_client.command(f"CREATE DATABASE IF NOT EXISTS {db}")
    print(db)

raw
trusted
gold


# 5. Criar tabela de metadados tracker_raw se nao existir

In [181]:
ddl_tracker = f"""
CREATE TABLE IF NOT EXISTS {TRACKER_DB}.{TRACKER_TABLE}
(
    system String,
    table_origin String,
    column_name String,
    column_type String,
    ordinal_position Int32,
    is_natural_key UInt8 DEFAULT 0,
    gold_table_name Nullable(String)
)
ENGINE = MergeTree()
ORDER BY (system, table_origin, ordinal_position)
"""
ch_client.command(ddl_tracker)
print(f"{TRACKER_DB}.{TRACKER_TABLE}")

default.tracker_raw


# 5b. Popular tracker_raw com lista de tabelas para raw (opcional)

Uma coluna placeholder por tabela. Substitua depois pelos metadados reais (colunas, tipos, chave natural).

In [182]:
TABLES_RAW = [
    "ginf.depara_cliente", "ginf.BASE_CEP_COMPLETA", "ginf.TST_CONTRATOS_BI", "ginf.BASE_REGIONAL",
    "ginf.TAB_CIDADE_DELITO_SP_CAP", "siga.SC5030", "siga.SC6030", "ginf.TST_HISTORICO_SOLICITACOES",
    "ginf.TST_SOLICIT_CADASTRADAS", "siga.SD2030", "siga.CN9030", "siga.SA1030", "siga.SA3030",
    "siga.SB1030", "siga.SZH030", "siga.SZJ030", "siga.SZU030", "siga.SZV030", "siga.SZW030",
    "siga.ZAA030", "siga.ZA1030", "siga.ZA3030", "siga.ZB3030", "siga.ZE8030", "siga.ZTX030",
    "siga.ZT1030", "siga.CN1030", "siga.CNB030", "siga.SE4030", "siga.SF2030", "ginf.TST_CONTRATOS",
    "scot.ERP_PRODUCT", "scot.ERP_PRODUCT_ITEM", "scot.ERP_VEHICLE", "scot.ERP_AGREEMENT",
    "scot.SC_CITY", "scot.SC_GROUP", "scot.SC_LOCATION", "scot.SC_REQ_FILE", "scot.SC_REQUISITION",
    "scot.SC_REQUISITION_HISTORY", "scot.SC_REQUISITION_QUEUE", "scot.SC_REQUISITION_STATUS",
    "scot.SC_RESERVE", "scot.SC_RESERVE_LOCATION", "scot.SC_RESULT_CODE", "scot.SC_ROLE", "scot.SC_STATE",
    "scot.CEPREG", "scot.SC_TASK", "scot.SC_WAREHOUSE", "scot.SC_TECHNICAL_REGISTER",
    "scot.SC_WEBSERVICE_REQUISITION", "scot.SC_WEBSERVICE_REQUISITION_HISTORY",
]
existing = set()
try:
    r = ch_client.query(f"SELECT system, table_origin FROM {TRACKER_DB}.{TRACKER_TABLE}")
    for row in r.result_rows:
        existing.add((str(row[0]).lower(), str(row[1]).lower()))
except Exception:
    pass
seen = set()
insert_rows = []
for full in TABLES_RAW:
    parts = full.split(".", 1)
    if len(parts) != 2:
        continue
    schema, table_origin = parts[0].strip().lower(), parts[1].strip()
    key = (schema, table_origin.lower())
    if key in seen or key in existing:
        continue
    seen.add(key)
    insert_rows.append((schema, table_origin, "_row_placeholder", "String", 1, 1, None))
if insert_rows:
    ch_client.insert(f"{TRACKER_DB}.{TRACKER_TABLE}", insert_rows,
        column_names=["system", "table_origin", "column_name", "column_type", "ordinal_position", "is_natural_key", "gold_table_name"])
print(len(insert_rows))

0


# 6. Ler tracker_raw e montar metadados por tabela

In [183]:
try:
    result = ch_client.query(
        f"SELECT system, table_origin, column_name, column_type, ordinal_position, "
        f"is_natural_key, gold_table_name FROM {TRACKER_DB}.{TRACKER_TABLE} "
        f"ORDER BY system, table_origin, ordinal_position"
    )
except Exception as e:
    raise RuntimeError(
        f"Falha ao ler {TRACKER_DB}.{TRACKER_TABLE}. Crie a tabela de metadados com colunas: "
        "system, table_origin, column_name, column_type, ordinal_position, is_natural_key, gold_table_name (opcional)."
    ) from e

col_names = ["system", "table_origin", "column_name", "column_type", "ordinal_position", "is_natural_key", "gold_table_name"]
rows = []
for row in result.result_rows:
    r = dict(zip(col_names, row))
    r["logical_name"] = f"{str(r.get('system', '')).lower()}_{str(r.get('table_origin', '')).lower()}"
    if r.get("is_natural_key") is None:
        r["is_natural_key"] = 0
    rows.append(r)

In [184]:
gold_col = "gold_table_name"

In [185]:
def build_table_metadata(rows):
    tables = {}
    for r in rows:
        key = (r["logical_name"], r["system"], r["table_origin"])
        if key not in tables:
            tables[key] = {
                "logical_name": r["logical_name"],
                "columns": [],
                "natural_key": [],
                "gold_table_name": None,
            }
        tbl = tables[key]
        col_def = (str(r["column_name"]).strip(), str(r["column_type"]).strip())
        tbl["columns"].append(col_def)
        if r.get("is_natural_key", 0) not in (0, None, False):
            tbl["natural_key"].append(str(r["column_name"]).strip())
        if gold_col and r.get(gold_col) and not tbl["gold_table_name"]:
            tbl["gold_table_name"] = str(r[gold_col]).strip().lower()
    return list(tables.values())

table_metas = build_table_metadata(rows)
print(len(table_metas))

54


# 7. Criar tabelas RAW

In [186]:
for meta in table_metas:
    name = meta["logical_name"]
    parts = [f"{c[0]} {c[1]}" for c in meta["columns"]]
    parts.append("snapshot_date Date")
    parts.append("ingestion_ts DateTime")
    order_cols = meta["natural_key"] if meta["natural_key"] else [meta["columns"][0][0]]
    order_by = ", ".join(order_cols)
    ddl = (
        f"CREATE TABLE IF NOT EXISTS raw.{name} ("
        + ", ".join(parts)
        + f") ENGINE = MergeTree() PARTITION BY snapshot_date ORDER BY ({order_by})"
    )
    ch_client.command(ddl)
    print(f"raw.{name}")

raw.ginf_base_cep_completa
raw.ginf_base_regional
raw.ginf_tab_cidade_delito_sp_cap
raw.ginf_tst_contratos
raw.ginf_tst_contratos_bi
raw.ginf_tst_historico_solicitacoes
raw.ginf_tst_solicit_cadastradas
raw.ginf_depara_cliente
raw.scot_cepreg
raw.scot_erp_agreement
raw.scot_erp_product
raw.scot_erp_product_item
raw.scot_erp_vehicle
raw.scot_sc_city
raw.scot_sc_group
raw.scot_sc_location
raw.scot_sc_requisition
raw.scot_sc_requisition_history
raw.scot_sc_requisition_queue
raw.scot_sc_requisition_status
raw.scot_sc_req_file
raw.scot_sc_reserve
raw.scot_sc_reserve_location
raw.scot_sc_result_code
raw.scot_sc_role
raw.scot_sc_state
raw.scot_sc_task
raw.scot_sc_technical_register
raw.scot_sc_warehouse
raw.scot_sc_webservice_requisition
raw.scot_sc_webservice_requisition_history
raw.siga_cn1030
raw.siga_cn9030
raw.siga_cnb030
raw.siga_sa1030
raw.siga_sa3030
raw.siga_sb1030
raw.siga_sc5030
raw.siga_sc6030
raw.siga_sd2030
raw.siga_se4030
raw.siga_sf2030
raw.siga_szh030
raw.siga_szj030
raw.siga_

# 8. Criar tabelas TRUSTED

In [187]:
for meta in table_metas:
    name = meta["logical_name"]
    parts = ["row_hash FixedString(64)"]
    parts += [f"{c[0]} {c[1]}" for c in meta["columns"]]
    parts.append("snapshot_date Date")
    parts.append("ingestion_ts DateTime")
    order_cols = meta["natural_key"] if meta["natural_key"] else [meta["columns"][0][0]]
    order_by = ", ".join(order_cols)
    ddl = (
        f"CREATE TABLE IF NOT EXISTS trusted.{name} ("
        + ", ".join(parts)
        + f") ENGINE = ReplacingMergeTree(snapshot_date) PARTITION BY snapshot_date ORDER BY ({order_by})"
    )
    ch_client.command(ddl)
    print(f"trusted.{name}")

trusted.ginf_base_cep_completa
trusted.ginf_base_regional
trusted.ginf_tab_cidade_delito_sp_cap
trusted.ginf_tst_contratos
trusted.ginf_tst_contratos_bi
trusted.ginf_tst_historico_solicitacoes
trusted.ginf_tst_solicit_cadastradas
trusted.ginf_depara_cliente
trusted.scot_cepreg
trusted.scot_erp_agreement
trusted.scot_erp_product
trusted.scot_erp_product_item
trusted.scot_erp_vehicle
trusted.scot_sc_city
trusted.scot_sc_group
trusted.scot_sc_location
trusted.scot_sc_requisition
trusted.scot_sc_requisition_history
trusted.scot_sc_requisition_queue
trusted.scot_sc_requisition_status
trusted.scot_sc_req_file
trusted.scot_sc_reserve
trusted.scot_sc_reserve_location
trusted.scot_sc_result_code
trusted.scot_sc_role
trusted.scot_sc_state
trusted.scot_sc_task
trusted.scot_sc_technical_register
trusted.scot_sc_warehouse
trusted.scot_sc_webservice_requisition
trusted.scot_sc_webservice_requisition_history
trusted.siga_cn1030
trusted.siga_cn9030
trusted.siga_cnb030
trusted.siga_sa1030
trusted.siga_

# 9. Criar tabelas GOLD (apenas com definicao explicita)

In [188]:
for meta in table_metas:
    gold_name = meta.get("gold_table_name") or ""
    if not gold_name:
        continue
    parts = [f"{c[0]} {c[1]}" for c in meta["columns"]]
    parts.append("snapshot_date Date")
    order_cols = meta["natural_key"] if meta["natural_key"] else [meta["columns"][0][0]]
    order_by = ", ".join(order_cols)
    ddl = (
        f"CREATE TABLE IF NOT EXISTS gold.{gold_name} ("
        + ", ".join(parts)
        + f") ENGINE = MergeTree() PARTITION BY toYYYYMM(snapshot_date) ORDER BY ({order_by})"
    )
    ch_client.command(ddl)
    print(f"gold.{gold_name}")

# 10. Resumo

In [189]:
for db in ["raw", "trusted", "gold"]:
    r = ch_client.query(f"SELECT name FROM system.tables WHERE database = '{db}'")
    names = [row[0] for row in r.result_rows]
    print(f"{db}: {len(names)} tabelas", names[:10] if len(names) > 10 else names)

raw: 54 tabelas ['ginf_base_cep_completa', 'ginf_base_regional', 'ginf_depara_cliente', 'ginf_tab_cidade_delito_sp_cap', 'ginf_tst_contratos', 'ginf_tst_contratos_bi', 'ginf_tst_historico_solicitacoes', 'ginf_tst_solicit_cadastradas', 'scot_cepreg', 'scot_erp_agreement']
trusted: 54 tabelas ['ginf_base_cep_completa', 'ginf_base_regional', 'ginf_depara_cliente', 'ginf_tab_cidade_delito_sp_cap', 'ginf_tst_contratos', 'ginf_tst_contratos_bi', 'ginf_tst_historico_solicitacoes', 'ginf_tst_solicit_cadastradas', 'scot_cepreg', 'scot_erp_agreement']
gold: 1 tabelas ['daily_sales_summary']
